<a href="https://colab.research.google.com/github/natdebandi/hate_speech_ar/blob/main/4_analysis_hate_ar.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Analisis del Dataset completo marzo 2020 - abril 2021 de X

**Natalia Dedandi**

Se aplica el clasificador seleccionado a todo el conjunto de datos recoelctado para analizar las características del odio en Argentina durante la pandemia

In [1]:
!pip install datasets seaborn


Recupero el conjutno completo de datos: https://huggingface.co/datasets/piuba-bigdata/articles_and_comments



In [2]:
from datasets import load_dataset
import pandas as pd

dataset = load_dataset("piubamas/articles_and_comments")


/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [3]:
dataset

DatasetDict({
    train: Dataset({
        features: ['tweet_id', 'text', 'title', 'url', 'user', 'body', 'created_at', 'comments'],
        num_rows: 537201
    })
})

In [4]:
ds1 = dataset["train"]

Ejemplo de una fila

In [5]:
ds1[122]

{'tweet_id': '1376979520729849858',
 'text': 'Una fuerte apuesta a la rueda de la agroindustrialización argentina https://t.co/1o6ThCdtxw https://t.co/TLfGhLBjHu',
 'title': 'Una fuerte apuesta a la rueda de la agroindustrialización argentina',
 'url': 'https://www.lanacion.com.ar/economia/campo/una-fuerte-apuesta-a-la-rueda-de-la-agroindustrializacion-argentina-nid30032021/?utm_term=Autofeed&utm_medium=Echobox&utm_source=Twitter#Echobox=1617130198',
 'user': 'LANACION',
 'body': 'Desde el primer día, YPF Agro se propuso proveer al campo argentino de los insumos y las energías necesarias para producir más y mejor, en un contexto donde la demanda de alimentos para la población mundial sigue creciendo.\n\nEl negocio de agro de YPF cerró 2020 con ventas mayores a 1.428 millones de dólares y más de 1,5 millones de toneladas de grano canjeadas, y apuesta a incrementar estos resultados en 2021.\n\nEn ese sentido, lanzó su nueva campaña de comunicación orientada a reforzar la idea de circular

En este conjunto de datos tengo las etiquetas de la clasificación realizada con el modelo BETO:

https://huggingface.co/piuba-bigdata/beto-contextualized-hate-speech

Puede usarse para comparar pero en principio nos interesa recuperar los comentarios completos para aplicarle el clasificador seleccionado

In [6]:
from datetime import datetime, timedelta
from tqdm.auto import tqdm
from collections import Counter

In [13]:
tweets_arg = []

for noticia in tqdm(dataset['train']):

    date_noticia = noticia["created_at"]
    # Convertimos la fecha de la noticia a un objeto de python
    datenoticia = datetime.strptime(date_noticia, "%Y-%m-%dT%H:%M:%S%fZ")
    i=0
    for comment in noticia["comments"]:
        date_comment = comment["created_at"]
        # Convertimos la fecha del comentario a un objeto de python
        datecomment = datetime.strptime(date_comment, "%Y-%m-%dT%H:%M:%S%fZ")
        # anexa comentarios de diarios argentinos
        tweets_arg.append({"id":i,
            "tweet_id_noticia": noticia["tweet_id"],
            "date_tweet": datecomment,
            "text": comment["text"],
            "CALLS":comment["prediction"]["CALLS"],
            "WOMEN":comment["prediction"]["WOMEN"],
            "LGBTI":comment["prediction"]["LGBTI"],
            "RACISM":comment["prediction"]["RACISM"],
            "CLASS":comment["prediction"]["CLASS"],
            "POLITICS":comment["prediction"]["POLITICS"],
            "DISABLED":comment["prediction"]["DISABLED"],
            "APPEARANCE":comment["prediction"]["APPEARANCE"],
            "CRIMINAL":comment["prediction"]["CRIMINAL"]
            })
        i=i+1

  0%|          | 0/537201 [00:00<?, ?it/s]

In [15]:
tweets_arg[11]

{'id': 3,
 'tweet_id_noticia': '1376940852011020288',
 'date_tweet': datetime.datetime(2021, 3, 30, 17, 49, 1, 500000),
 'text': '@LANACION Re largo el sueño',
 'CALLS': 0,
 'WOMEN': 0,
 'LGBTI': 0,
 'RACISM': 0,
 'CLASS': 0,
 'POLITICS': 0,
 'DISABLED': 0,
 'APPEARANCE': 0,
 'CRIMINAL': 0}

In [16]:
# prompt: create a dataframe from tweets_arg

df_tweets_arg = pd.DataFrame(tweets_arg)


In [19]:
df_tweets_arg['HATEFUL']= df_tweets_arg[['CALLS','WOMEN','LGBTI','RACISM','CLASS','POLITICS','DISABLED','APPEARANCE','CRIMINAL']].max(axis=1)

In [20]:
df_tweets_arg.head()

,id,tweet_id_noticia,date_tweet,text,CALLS,WOMEN,LGBTI,RACISM,CLASS,POLITICS,DISABLED,APPEARANCE,CRIMINAL,HATEFUL
0,0,1376940813968609288,2021-03-30 17:03:00.900,@clarincom A mi me preocupa el trabajo.. La ev...,0,0,0,0,0,0,0,0,0,0
1,1,1376940813968609288,2021-03-30 17:05:04.500,@clarincom Lo que preocupa. https://t.co/Vmf9V...,0,0,0,0,0,0,0,0,0,0
2,2,1376940813968609288,2021-03-30 17:06:03.100,@clarincom Lo que les preocupa. https://t.co/P...,0,0,0,0,0,0,0,0,0,0
3,3,1376940813968609288,2021-03-30 17:11:02.300,@clarincom Le recomendaríamos al presidente de...,0,0,0,0,0,0,0,0,0,0
4,4,1376940813968609288,2021-03-30 17:26:00.600,@clarincom Para salvar al correo de la quiebra...,0,0,0,0,0,0,0,0,0,0


coloco el mismo orden de las etiquetas que usa el clasificador

In [21]:
# prompt: change the order of columns of df_tweets_arg and put HATEFUL before CALLS

df_tweets_arg = df_tweets_arg[['id', 'tweet_id_noticia', 'date_tweet', 'text', 'HATEFUL', 'CALLS', 'WOMEN',
       'LGBTI', 'RACISM', 'CLASS', 'POLITICS', 'DISABLED', 'APPEARANCE',
       'CRIMINAL']]
df_tweets_arg.head()


,id,tweet_id_noticia,date_tweet,text,HATEFUL,CALLS,WOMEN,LGBTI,RACISM,CLASS,POLITICS,DISABLED,APPEARANCE,CRIMINAL
0,0,1376940813968609288,2021-03-30 17:03:00.900,@clarincom A mi me preocupa el trabajo.. La ev...,0,0,0,0,0,0,0,0,0,0
1,1,1376940813968609288,2021-03-30 17:05:04.500,@clarincom Lo que preocupa. https://t.co/Vmf9V...,0,0,0,0,0,0,0,0,0,0
2,2,1376940813968609288,2021-03-30 17:06:03.100,@clarincom Lo que les preocupa. https://t.co/P...,0,0,0,0,0,0,0,0,0,0
3,3,1376940813968609288,2021-03-30 17:11:02.300,@clarincom Le recomendaríamos al presidente de...,0,0,0,0,0,0,0,0,0,0
4,4,1376940813968609288,2021-03-30 17:26:00.600,@clarincom Para salvar al correo de la quiebra...,0,0,0,0,0,0,0,0,0,0


In [ ]:
# prompt: save df_tweets_arg to csv file

df_tweets_arg.to_csv('data/tweets_arg.csv', index=False)


In [40]:
# prompt: len of df_tweets_arg

len(df_tweets_arg)


8569648